# KuaiRand-27K max_seq_len sweep — pulled from wandb cloud

Same analysis as `analyze_seqlen_sweep.ipynb`, but data comes from the **wandb public API**
(`ngocjr7/hstu`) instead of local `wandb-summary.json` files — so it runs on any host with
network + a wandb login (no `/checkpoints` mount needed).

Runs: `kr27k_len<L>_s<S>`, L ∈ {512…16384} × seeds {1,2,3}, 1 epoch. Final end-of-epoch eval only.

**Auth:** needs `WANDB_API_KEY` in the env (or a prior `wandb login`).

In [ ]:
import os, re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# Bento-Plotly registered template (sets pio.templates.default + add_source + OKABE_ITO)
exec(open(os.path.expanduser(
    "/home/ngocbh/.claude/agent-market/plugins/10x-data-scientist/skills/visualization/bento-plotly/references/bento_style_template.py"
)).read())

In [ ]:
# --- Pull final eval metrics from the wandb cloud API ---
import wandb

ENTITY = os.environ.get("WANDB_ENTITY") or "ngocjr7"
PROJECT = os.environ.get("WANDB_PROJECT", "hstu")
NAME_RE = re.compile(r"^kr27k_len(\d+)_s(\d+)$")

api = wandb.Api()
# Server-side filter to the sweep's runs; Python regex is the final guard.
runs = api.runs(f"{ENTITY}/{PROJECT}",
                filters={"display_name": {"$regex": r"^kr27k_len\d+_s\d+$"}})

rows = []
for run in runs:
    m = NAME_RE.match(run.name)
    if not m:
        continue
    seq_len, seed = int(m.group(1)), int(m.group(2))
    summ = run.summary
    for k in summ.keys():
        mm = re.match(r"eval_metric/lifetime_(gauc|ne|gauc_num_samples)/(.+)$", k)
        if mm:
            rows.append(dict(seq_len=seq_len, seed=seed, metric=mm.group(1),
                             task=mm.group(2), value=summ[k],
                             run_id=run.id, state=run.state))

df = pd.DataFrame(rows)
assert not df.empty, f"no kr27k_len* runs found in {ENTITY}/{PROJECT}"

# Prefer finished runs; if a (len,seed) was rerun, keep the most recent run_id.
if (df.state == "finished").any():
    df = df[df.state == "finished"]
df = (df.sort_values("run_id")
        .drop_duplicates(["seq_len", "seed", "metric", "task"], keep="last")
        .drop(columns=["run_id", "state"]))

SEQLENS = sorted(df.seq_len.unique())
print("(seq_len, seed) cells loaded:", df[['seq_len', 'seed']].drop_duplicates().shape[0])
print("seq_lens:", SEQLENS)

nsamp = (df[df.metric == "gauc_num_samples"].groupby("task").value.mean().sort_values(ascending=False))
WELL_POWERED = nsamp[nsamp >= 800].index.tolist()
print("\nmean eval sessions per task:")
print(nsamp.round(0).to_string())
print("\nwell-powered tasks (n>=800):", WELL_POWERED)

In [ ]:
def hex_to_rgba(h, a):
    h = h.lstrip("#")
    r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{a})"

def sweep_figure(metric, ylabel, title_text, sub_text, peak_task, peak_x, peak_label):
    """Line-per-task chart of `metric` vs seq_len, mean of 3 seeds with ±1σ band."""
    agg = (df[df.metric == metric]
           .query("task in @WELL_POWERED")
           .groupby(["task", "seq_len"]).value.agg(["mean", "std"]).reset_index())
    fig = go.Figure()
    for i, task in enumerate(WELL_POWERED):  # already ordered by sample count desc
        sub = agg[agg.task == task].sort_values("seq_len")
        color = OKABE_ITO[i]
        x = sub.seq_len.tolist()
        mean = sub["mean"].tolist()
        std = sub["std"].fillna(0).tolist()
        upper = [m + s for m, s in zip(mean, std)]
        lower = [m - s for m, s in zip(mean, std)]
        fig.add_trace(go.Scatter(
            x=x + x[::-1], y=upper + lower[::-1], fill="toself",
            fillcolor=hex_to_rgba(color, 0.12), line=dict(width=0),
            hoverinfo="skip", showlegend=False))
        fig.add_trace(go.Scatter(
            x=x, y=mean, mode="lines+markers", name=task,
            line=dict(color=color, width=2.5), marker=dict(size=7), customdata=std,
            hovertemplate=(f"<b>{task}</b><br>seq_len=%{{x:,}} tokens<br>"
                           f"{ylabel}=%{{y:.4f}} (±%{{customdata:.4f}})<extra></extra>")))
    pk = agg[(agg.task == peak_task) & (agg.seq_len == peak_x)]["mean"].iloc[0]
    fig.add_annotation(x=np.log10(peak_x), y=pk, text=peak_label, showarrow=True,
                       arrowhead=2, ax=0, ay=-32, font=dict(size=11, color="#555555"))
    fig.update_layout(
        title=dict(text=f"<b>{title_text}</b><br><sub>{sub_text}</sub>"),
        xaxis=dict(title="max sequence length (tokens, log₂)", type="log",
                   tickvals=SEQLENS, ticktext=[f"{s:,}" for s in SEQLENS]),
        yaxis=dict(title=ylabel),
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
        margin=dict(r=200))
    add_source(fig, f"wandb {ENTITY}/{PROJECT} · KuaiRand-27K seq-len sweep · 1 epoch · "
                    "end-of-epoch eval · low-volume tasks (n<250) excluded")
    return fig

In [ ]:
# --- gAUC: higher is better ---
fig_gauc = sweep_figure(
    metric="gauc", ylabel="eval gAUC (per-user AUC)",
    title_text="Longer histories lift click & long-view ranking",
    sub_text="Eval gAUC vs HSTU max sequence length · KuaiRand-27K · mean of 3 seeds ±1σ",
    peak_task="long_view", peak_x=8192, peak_label="gains plateau @ 8K")
fig_gauc.add_hline(y=0.5, line_dash="dot", line_color="#999999",
                   annotation_text="random (0.50)", annotation_position="bottom left")
fig_gauc.show()

In [ ]:
# --- NE: lower is better (normalized entropy / calibrated loss) ---
fig_ne = sweep_figure(
    metric="ne", ylabel="eval NE (lower = better)",
    title_text="Calibrated loss improves on high-volume tasks",
    sub_text="Eval NE vs HSTU max sequence length · KuaiRand-27K · mean of 3 seeds ±1σ",
    peak_task="is_click", peak_x=4096, peak_label="best @ 4K–16K")
fig_ne.show()